Le data management sur des données de retail

In [2]:
from pyspark.sql.types import *
import pyspark.sql.functions as sf
import string
from pyspark.sql.window import Window
from pyspark.sql.types import * 

In [3]:
# les paramètres du pgm  
selected_columns = ["ID_CLIENT", "DT_CMDE","QTE_ART", "CA_TOTAL", "CA_PROMO", "COLLECTION","FLAG_SOLDE","TICKET","CANAL_ACHAT"]
root = "/FileStore/tables/DataSource/data-management/retail"
path_trx = "{0}/{1}".format(root,"transactions.csv")
path_client = "{0}/{1}".format(root,"clients.csv")
date_ref_min = "2012-01-01"
date_ref_max = "2013-01-01"

In [4]:
# chargement des données 
df = spark.read.csv(path_trx,sep = ";",header = True,inferSchema=True)


In [5]:
# conversion DT_CMDE en date 
df = df.withColumn("DT_CMDE",sf.to_date("DT_CMDE","dd/MM/yyyy"))

In [6]:
# vérification si on n'a pas des valeures null apèrs la conversion 
df.filter(sf.col("DT_CMDE").isNull()).count()

Out[5]: 0

In [7]:
# selection des colonnes 
df = df.select(*selected_columns)

In [8]:
# Calcul de la variable année, mois , jour 
df = df.withColumn("DT_CMDE_YEAR",sf.year("DT_CMDE")) 
df = df.withColumn("DT_CMDE_MONTH",sf.month("DT_CMDE"))
df = df.withColumn("DT_CMDE_DAY",sf.dayofmonth("DT_CMDE"))

In [9]:
df = df.filter(sf.col("CA_TOTAL") >0)

In [10]:
# filtre par rapport une date min et une date max 
df = df.filter(sf.col("DT_CMDE").between(date_ref_min,date_ref_max))

In [11]:
# Calcul de variable trimestre 
df = df.withColumn("Trimestre",sf.when(sf.col("DT_CMDE_MONTH").between(1,4),"T1").\
                                       when(sf.col("DT_CMDE_MONTH").between(4,7),"T2").\
                                       when(sf.col("DT_CMDE_MONTH").between(7,10),"T3").\
                   
                                       when(sf.col("DT_CMDE_MONTH").between(10,13),"T4"))
                                      

Calcul de nouveau indicateur

In [13]:
# Création de fenetre de partitionnement par rapport au client 
partition = Window.partitionBy("ID_CLIENT").orderBy('DT_CMDE')

In [14]:
# décalage des ligne par rapport à la fenetre partitionnement 
df = df.withColumn('DT_CMDE_PRV',sf.lag(df['DT_CMDE']).over(partition))

In [15]:
# calcul de la variable de delai inter achat en jours et en mois 
df = df.withColumn("DELAI_DAYS",sf.datediff('DT_CMDE','DT_CMDE_PRV'))
df = df.withColumn("DELAI_MONTH",sf.months_between('DT_CMDE','DT_CMDE_PRV'))

In [16]:
# Somme cumulée de CA par client
df = df.withColumn("SUM_CA_CUM",sf.sum("CA_TOTAL").over(partition))

In [17]:
# Création de fenetre de partitionnement par rapport au client et canal d'achat
partition = Window.partitionBy("ID_CLIENT","CANAL_ACHAT").orderBy('DT_CMDE')


In [18]:
df = df.withColumn('DT_CMDE_PRV_CANAL',sf.lag(df['DT_CMDE']).over(partition))

In [19]:
# calcul de la variable de delai inter achat en jours et en mois 
df = df.withColumn("DELAI_DAYS_CANAL",sf.datediff('DT_CMDE','DT_CMDE_PRV_CANAL'))
df = df.withColumn("DELAI_MONTH_CANAL",sf.months_between('DT_CMDE','DT_CMDE_PRV_CANAL'))

Construction de la table des agrégats des clients + Analyse du delai inter achat par canal

In [21]:
# définition des agrégats à calculer 
expr_agg = [sf.min("DELAI_MONTH").alias("MIN_DELAI_MONTH"),
            sf.max("DELAI_MONTH").alias("MAX_DELAI_MONTH"),
            sf.avg("DELAI_MONTH").alias("AVG_DELAI_MONTH"),
            sf.stddev("DELAI_MONTH").alias("STD_DELAI_MONTH"),
            sf.count("TICKET").alias("NB_VISIT"),
            sf.count(sf.when(df["FLAG_SOLDE"] == "1",1)).alias("TOTAL_VISIT_SOLDE"),
            sf.count(sf.when(df["CANAL_ACHAT"] == "STORE",1)).alias("TOTAL_VISIT_STORE"),
            sf.count(sf.when(df["CANAL_ACHAT"] == "WEB",1)).alias("TOTAL_VISIT_WEB"),
            sf.count(sf.when(df["FLAG_SOLDE"] == "1",1)).alias("TOTAL_VISIT_SOLDE"),sf.count(sf.when(df["CANAL_ACHAT"] == "STORE",1)).alias("TOTAL_VISIT_STORE"),
            sf.count(sf.when(df["CANAL_ACHAT"] == "WEB",1)).alias("TOTAL_VISIT_WEB"),
            sf.sum("CA_TOTAL").alias("TOTAL_CA"),
            sf.sum(sf.when(df["CANAL_ACHAT"] == "WEB",df["CA_TOTAL"])).alias("TOTAL_CA_WEB"),
            sf.sum(sf.when(df["CANAL_ACHAT"] == "STORE",df["CA_TOTAL"])).alias("TOTAL_CA_STORE"),
            sf.sum("QTE_ART").alias("TOTAL_QTY"),
            sf.sum(sf.when(df["CANAL_ACHAT"] == "WEB",df["QTE_ART"])).alias("TOTAL_QTY_WEB"),
            sf.sum(sf.when(df["CANAL_ACHAT"] == "STORE",df["QTE_ART"])).alias("TOTAL_QTY_STORE")]

In [22]:
# Calcul du data frame d'agrégation 
agg_clients = df.groupBy("ID_CLIENT").agg(*expr_agg)


In [23]:
# Construction de la table client avec leur delai moyen,min,max par canal 
df.groupBy("ID_CLIENT").pivot("CANAL_ACHAT").agg(sf.avg("DELAI_MONTH_CANAL").alias("AVG_DELAI"),\
                                                 sf.min("DELAI_MONTH_CANAL").alias("MIN_DELAI"),\
                                                 sf.max("DELAI_MONTH_CANAL").alias("MAX_DELAI"),\
                                                ).filter(sf.col("WEB_MIN_DELAI").isNotNull()).show()

+----------+------------------+---------------+---------------+-------------+-------------+-------------+
 ID_CLIENT| STORE_AVG_DELAI|STORE_MIN_DELAI|STORE_MAX_DELAI|WEB_AVG_DELAI|WEB_MIN_DELAI|WEB_MAX_DELAI|
+----------+------------------+---------------+---------------+-------------+-------------+-------------+
8210001760| null| null| null| 8.70967742| 8.70967742| 8.70967742|
8210003423| null| null| null| 1.38709677| 1.38709677| 1.38709677|
9002007142| null| null| null| 0.32258065| 0.32258065| 0.32258065|
8210002049| null| null| null| 3.12903226| 3.12903226| 3.12903226|
9000135614| 3.0| 3.0| 3.0| 0.48387097| 0.48387097| 0.48387097|
8500356637|1.5268817200000002| 0.32258065| 3.35483871| 0.58064516| 0.35483871| 0.80645161|
9002007336|0.5913978483333334| 0.0| 1.64516129| 0.5483871| 0.5483871| 0.5483871|
+----------+------------------+---------------+---------------+-------------+-------------+-------------+

Ajout des information client à la table des agrégats des  clients

In [25]:
# chargement des données client 
clients = spark.read.csv(path_client,sep = ";",header = True,timestampFormat ="dd/MM/yyyy",inferSchema = True)
clients.printSchema()

root
-- ID_CLIENT: long (nullable = true)
-- CIVILITE: integer (nullable = true)
-- DATE_NAIS: timestamp (nullable = true)
-- DATE_CREATION: timestamp (nullable = true)
-- CODE_POSTAL: string (nullable = true)
-- FLAG_EMAIL: integer (nullable = true)
-- FLAG_MOBILE: integer (nullable = true)
-- NB_ENFANT: string (nullable = true)
-- OPTIN_EMAIL: string (nullable = true)
-- OPTIN_SMS: string (nullable = true)
-- FLAG_ADRESSE: string (nullable = true)
-- NPAI_ADRESSE: string (nullable = true)

In [26]:
# selection des colonnes 
clients = clients.select("ID_CLIENT","CIVILITE",sf.to_date("DATE_NAIS").alias("DATE_NAIS"),sf.to_date("DATE_CREATION").alias("DATE_CREATION"),"CODE_POSTAL")


In [27]:
# conversion de la date de ref en spark column 
date_ref = sf.to_date(sf.lit((date_ref_max)))


In [28]:
# Regroupement de la civilité 
clients = clients.withColumn("CIVILITE",sf.when(sf.col("CIVILITE") == 1,"M").when(sf.col("CIVILITE").isin(2,3),"Mme").otherwise(None))


In [29]:
# calcul l'ecart en mois entre la date de réference (la dateref) et Date de Naissance et date de création 
clients = clients.withColumn("Age",sf.months_between(date_ref,"DATE_NAIS")/12)
clients = clients.withColumn("Anciennete",sf.months_between(date_ref,"DATE_CREATION"))

In [30]:
# Discrétisation de l'Anciennete
clients = clients.withColumn("Anciennete_ENC",sf.when(sf.col("Anciennete") <12,1).when(sf.col("Anciennete") < 24,2).otherwise(3))

In [31]:
clients = clients.join(agg_clients,"ID_CLIENT","inner")
clients.count()

Out[27]: 6775

Création de groupe de client

In [33]:
# aggrégation de la table des transaction au niveau client, Trimestre et année 
df_client_trimestre = df.groupBy("ID_CLIENT","Trimestre","DT_CMDE_YEAR").agg(sf.sum("CA_TOTAL").alias("CA_Trimestre"))

In [34]:
# creation d'une fonction qui calcul les quartiles dsq
import numpy as np
def percentile(list_of_values,proba):
  values = np.array(list_of_values)
  p = np.percentile(values, proba) # return 50th percentile, e.g median.
  return float(p)

In [35]:
# creation des udfs qui calcul q1,q2,q3, d'une colonne de liste 
q1 = sf.udf(lambda x:percentile(x,25),FloatType())
q2 = sf.udf(lambda x:percentile(x,50),FloatType())
q3 = sf.udf(lambda x:percentile(x,75),FloatType())

In [36]:
# aggrégation de la table des transaction au niveau année  et Trimestre
df_trimestre = df_client_trimestre.groupBy("Trimestre","DT_CMDE_YEAR").agg(sf.collect_list("CA_Trimestre").alias("list_ca"))

In [37]:
# Calcul des quartile de CA de chauqe trimestre
df_trimestre = df_trimestre.withColumn("Q1",q1("list_ca"))
df_trimestre = df_trimestre.withColumn("Q2",q2("list_ca"))
df_trimestre = df_trimestre.withColumn("Q3",q3("list_ca"))

In [38]:
df_trimestre.show()

+---------+------------+--------------------+------+-----+-------+
Trimestre|DT_CMDE_YEAR| list_ca| Q1| Q2| Q3|
+---------+------------+--------------------+------+-----+-------+
 T3| 2012|[69.0, 72.25, 139...| 55.0| 90.3| 159.0|
 T1| 2012|[46.0, 119.3, 26....|48.375| 80.2|146.325|
 T2| 2012|[109.0, 90.5, 36....| 44.9| 76.3|134.575|
 T4| 2012|[336.0, 27.3, 206...| 61.0|99.15| 170.25|
+---------+------------+--------------------+------+-----+-------+

In [39]:
df_client_trimestre = df_client_trimestre.\
join(df_trimestre,(df_client_trimestre.Trimestre==df_trimestre.Trimestre) & (df_client_trimestre.DT_CMDE_YEAR == df_trimestre.DT_CMDE_YEAR),"inner").\
drop(df_trimestre.DT_CMDE_YEAR).drop(df_trimestre.Trimestre)

In [40]:
# Construction des catégorie de client par trimestre  
df_client_trimestre = df_client_trimestre.withColumn("Segment_YEAR",sf.when(sf.col("CA_Trimestre") <= sf.col("Q1"),1)\
                                          .when(sf.col("CA_Trimestre") <= sf.col("Q2"),2)\
                                          .otherwise(3))

In [41]:
# Agrégation des catégories au niveau client 
df_client = df_client_trimestre.groupBy("ID_CLIENT").agg(sf.max("Segment_YEAR").alias("Segment"))

In [42]:
df_client_trimestre.groupby("Segment_YEAR").count().show()

+------------+-----+
Segment_YEAR|count|
+------------+-----+
 1| 2650|
 3| 5221|
 2| 2592|
+------------+-----+

In [43]:
clients = clients.join(df_client,"ID_CLIENT","inner")

statistique descriptive des agrégat

In [45]:
df_ca = clients.select("TOTAL_CA")

In [46]:
# stat desc de la variable Total CA 
df_ca.describe().show()

+-------+------------------+
summary| TOTAL_CA|
+-------+------------------+
 count| 6775|
 mean|187.71112189655528|
 stddev|243.61933596496024|
 min| 5.0|
 max| 5109.7|
+-------+------------------+

In [47]:

clients.approxQuantile("TOTAL_CA",[0.25,0.5,0.75],0)

Out[41]: [59.84, 116.3, 224.45]

In [48]:
# table croisée entre l'ancienneté et le segment de CA 
clients.stat.crosstab("Segment","Anciennete_ENC").show()

+----------------------+----+---+----+
Segment_Anciennete_ENC| 1| 2| 3|
+----------------------+----+---+----+
 2| 719|203| 731|
 1| 635|176| 562|
 3|1318|588|1843|
+----------------------+----+---+----+

Manupulation SQL

In [50]:
# Creation de la table sql 
clients.createTempView("clients_sql")

In [51]:
# requete en sql des clients
sql_df = spark.sql("select * from clients_sql ")

In [52]:
# mise en place de udf sql 
div_spark = sf.udf(lambda x,y: float(x)/float(y) if y !=0 else 0.0,FloatType())
spark.udf.register("pannier_moyen",div_spark)

Out[45]: <function __main__.<lambda>(x, y)>

In [53]:
# requete avec operateur pannier moyen 
sql_df = spark.sql("select *, pannier_moyen(TOTAL_CA,TOTAL_QTY) as Pannier_Moyen from clients_sql")

In [54]:
sql_df.show()

+----------+--------+----------+-------------+-----------+------------------+-----------+--------------+---------------+---------------+------------------+------------------+--------+-----------------+-----------------+---------------+-----------------+-----------------+---------------+------------------+------------+------------------+---------+-------------+---------------+-------+-------------+
 ID_CLIENT|CIVILITE| DATE_NAIS|DATE_CREATION|CODE_POSTAL| Age| Anciennete|Anciennete_ENC|MIN_DELAI_MONTH|MAX_DELAI_MONTH| AVG_DELAI_MONTH| STD_DELAI_MONTH|NB_VISIT|TOTAL_VISIT_SOLDE|TOTAL_VISIT_STORE|TOTAL_VISIT_WEB|TOTAL_VISIT_SOLDE|TOTAL_VISIT_STORE|TOTAL_VISIT_WEB| TOTAL_CA|TOTAL_CA_WEB| TOTAL_CA_STORE|TOTAL_QTY|TOTAL_QTY_WEB|TOTAL_QTY_STORE|Segment|Pannier_Moyen|
+----------+--------+----------+-------------+-----------+------------------+-----------+--------------+---------------+---------------+------------------+------------------+--------+-----------------+-----------------+---------------+-----------------+-----------------+---------------+------------------+------------+------------------+---------+-------------+---------------+-------+-------------+
 31006521| Mme|1991-07-25| 2012-10-06| 67000|21.435483870833334| 2.83870968| 1| 2.4516129| 2.4516129| 2.4516129| NaN| 2| 0| 2| 0| 0| 2| 0| 198.0| null| 198.0| 2| null| 2| 3| 99.0|
 90356582| Mme|1986-11-26| 2009-04-11| 91220|26.099462365833332|44.67741935| 3| 0.0| 0.0| 0.0| NaN| 2| 2| 2| 0| 2| 2| 0| 239.43| null| 239.43| 7| null| 7| 3| 34.204285|
 111001837| Mme| null| 2011-08-08| 75015| null|16.77419355| 2| 0.03225806| 6.67741935| 3.354838705| 4.698838610237345| 3| 2| 3| 0| 2| 3| 0| 378.4| null| 378.4| 8| null| 8| 3| 47.3|
 241013592| Mme| null| 2012-10-02| 06480| null| 2.96774194| 1| 0.74193548| 0.74193548| 0.74193548| NaN| 2| 0| 2| 0| 0| 2| 0| 183.3| null| 183.3| 2| null| 2| 3| 91.65|
 261010985| Mme| null| 2012-06-13| 92310| null| 6.61290323| 1| null| null| null| null| 1| 0| 1| 0| 0| 1| 0| 100.0| null| 100.0| 2| null| 2| 3| 50.0|
 291020456| Mme| null| 2012-11-29| 80000| null| 1.09677419| 1| null| null| null| null| 1| 0| 1| 0| 0| 1| 0| 69.0| null| 69.0| 1| null| 1| 2| 69.0|
 462010675| Mme| null| 2012-02-22| 94160| null|10.32258065| 1| 0.12903226| 3.93548387|1.7338709675000001|1.6008961667734112| 5| 1| 5| 0| 1| 5| 0| 305.95| null| 305.95| 6| null| 6| 3| 50.991665|
 462011138| Mme| null| 2012-04-20| 92370| null| 8.38709677| 1| null| null| null| null| 1| 0| 1| 0| 0| 1| 0| 35.0| null| 35.0| 1| null| 1| 1| 35.0|
 471000862| Mme|1987-02-18| 2010-03-29| 13100|25.870967741666664|33.09677419| 3| null| null| null| null| 1| 0| 1| 0| 0| 1| 0| 75.0| null| 75.0| 1| null| 1| 2| 75.0|
 551003322| Mme| null| 2010-10-06| 21000| null|26.83870968| 3| 1.32258065| 2.58064516| 1.951612905|0.8895859461911312| 3| 1| 3| 0| 1| 3| 0| 246.9| null| 246.9| 4| null| 4| 3| 61.725|
 611001299| Mme|1964-12-06| 2010-09-07| 10190| 48.06989247333333|27.80645161| 3| null| null| null| null| 1| 1| 1| 0| 1| 1| 0| 39.0| null| 39.0| 1| null| 1| 1| 39.0|
 671002445| Mme|1979-07-02| 2012-09-25| 61100| 33.49731182833333| 3.22580645| 1| 2.06451613| 2.06451613| 2.06451613| NaN| 2| 0| 2| 0| 0| 2| 0| 240.3| null| 240.3| 2| null| 2| 3| 120.15|
1502008535| Mme| null| 2011-11-08| 75011| null|13.77419355| 2| null| null| null| null| 1| 1| 1| 0| 1| 1| 0| 73.8| null| 73.8| 3| null| 3| 2| 24.6|
8091000153| Mme|1969-04-15| 2010-01-02| 60112| 43.71236559166667|35.96774194| 3| null| null| null| null| 1| 1| 1| 0| 1| 1| 0| 46.8| null| 46.8| 2| null| 2| 1| 23.4|
8091001885| Mme|1989-11-21| 2010-11-27| 60112|23.112903225833335|25.16129032| 3| 2.90322581| 2.90322581| 2.90322581| NaN| 2| 1| 2| 0| 1| 2| 0| 133.8| null| 133.8| 4| null| 4| 3| 33.45|
8101000560| Mme|1953-07-07| 2010-08-02| 60200|59.483870967499996|28.96774194| 3| 0.0| 1.25806452|0.3924731179166667| 0.405105484866419| 25| 5| 25| 0| 5| 25| 0| 2085.37| null| 2085.37| 51| null| 51| 3| 40.889606|
8261002756| Mme| null| 2011-10-15| 26740| null| 14.5483871| 2| 1.03225806| 2.74193548|